# CUP Farmland Analysis 🧑‍🌾🐄🌽

A screening tool for evaluating Conditional Use Permit applications against California Important Farmland (FMMP) designations in Imperial County. Given a set of parcel APNs, it dissolves them into an area of interest, clips the FMMP layer to that AOI, and reports the acreage and percentage of the proposal that sits on designated farmland.

#### Inputs

- **APNs** — list of parcel identifiers defining the CUP footprint
- **CUP name** — label used in the map title and exported analysis

#### Outputs

- Interactive Folium map (`cup_farmland_analysis.html`) with toggleable layers, satellite and street basemaps, and an embedded analysis summary
- AOI acreage, farmland acreage, and percent coverage

In [ ]:
#@title CUP Farmland Analysis Inputs { display-mode: "form" }
APNs_raw = "039-140-013, 039-140-014" #@param {type:"string"}
CUP_number = "CUP #22-0030" #@param {type:"string"}
CUP_name = "Northstar 2 Solar Energy Generation and Battery Storage" #@param {type:"string"}

APNs = [a.strip() for a in APNs_raw.replace("\n", ",").split(",") if a.strip()]

print(f"APNs ({len(APNs)}): {APNs}")
print(f"CUP number: {CUP_number!r}")
print(f"Map title: {CUP_name!r}")

In [ ]:
#!pip install arcgis arcgis-mapping

## Load Layers
Imperial County Parcels
(https://services7.arcgis.com/RomaVqqozKczDNgd/arcgis/rest/services/Imperial_County_Parcels_(View)/FeatureServer)

California Farmland
(https://gis.conservation.ca.gov/server/rest/services/DLRP/CaliforniaImportantFarmland_mostrecent/MapServer)

In [ ]:
# load farmland
ca_farmland_url = "https://gis.conservation.ca.gov/server/rest/services/DLRP/CaliforniaImportantFarmland_mostrecent/MapServer"
cup_parcels_url = "https://services7.arcgis.com/RomaVqqozKczDNgd/arcgis/rest/services/Imperial_County_Parcels_(View)/FeatureServer"

In [ ]:
# load or create layers
apns_formatted = ", ".join(f"'{apn}'" for apn in APNs)
p = load_arcgis_feature_layer(cup_parcels_url, where=f"APN IN ({apns_formatted})")
farmland = load_arcgis_feature_layer(ca_farmland_url,
                where=f"County = 'Imperial' AND Code NOT IN ({NOT_FARMLAND})")
aoi = dissolve_features(p, name=CUP_name) # create cup area of interest

results = farmland_coverage(aoi, farmland)

In [ ]:
# cup aoi symbology
thick_red_outline = line_symbol("#fc0303", width=2.5)
no_fill_thick_red_outline = fill_symbol(opacity=0.0, outline=thick_red_outline)

# parcels symbology
thin_red_outline = line_symbol("#fc0303", width=0.4)
no_fill_thin_red_outline = fill_symbol(opacity=0.0, outline=thin_red_outline)

# farmland symbology
green_outline = line_symbol("#008f1d", width=1, opacity=0.5)
green_fill_green_outline = fill_symbol(color="#008f1d",
                                       opacity=0.25,
                                       outline=green_outline)

# aoi farmland symbology
aoi_outline = line_symbol("#008f1d", width=2)
aoi_green_fill_green_outline = fill_symbol(color="#008f1d",
                                       opacity=0.5,
                                       outline=aoi_outline)

# Map
map = gis.map()

# farmland
map.content.add(farmland,
                popup_info=PopupInfo(title= "ID: {OBJECTID}",
                                     description = "Type: {Code}"),
                drawing_info=simple_renderer(green_fill_green_outline)
                )

# Only add the clipped results if there are intersecting features
if results['clipped'].features:
    map.content.add(results['clipped'],
                    drawing_info=simple_renderer(aoi_green_fill_green_outline))

# parcels
map.content.add(p, drawing_info=simple_renderer(no_fill_thin_red_outline))

# cup aoi
map.content.add(aoi, drawing_info=simple_renderer(no_fill_thick_red_outline))

map.zoom_to_layer(aoi)

map

In [ ]:
#### Export to HTML
import folium

def fset_to_geojson(fset):
    """Convert an esri FeatureSet to GeoJSON dict."""
    return {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "geometry": {"type": "Polygon", "coordinates": f.geometry["rings"]},
                "properties": {k: v for k, v in f.attributes.items()},
            }
            for f in fset.features
        ],
    }

# Center on AOI centroid
ring = aoi.features[0].geometry["rings"][0]
center_lat = sum(pt[1] for pt in ring) / len(ring)
center_lon = sum(pt[0] for pt in ring) / len(ring)

fm = folium.Map(location=[center_lat, center_lon], zoom_start=15)

# Basemaps (toggleable; last added is the default)

#folium.TileLayer("CartoDB Voyager", name="Light").add_to(fm)
folium.TileLayer(
    tiles="https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics", name="Satellite"
).add_to(fm)
folium.TileLayer(
    tiles="https://services.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Topo"
).add_to(fm)

# Farmland (background)
folium.GeoJson(
    fset_to_geojson(farmland), name="Farmland (FMMP)",
    style_function=lambda x: {"fillColor": "#008f1d", "color": "#008f1d",
                              "weight": 1, "fillOpacity": 0.25},
).add_to(fm)

# Farmland within AOI (highlighted)
folium.GeoJson(
    fset_to_geojson(results["clipped"]), name="Farmland within AOI",
    style_function=lambda x: {"fillColor": "#008f1d", "color": "#005c13",
                              "weight": 2, "fillOpacity": 0.55},
).add_to(fm)

# Parcels (thin red)
folium.GeoJson(
    fset_to_geojson(p), name="CUP parcels",
    style_function=lambda x: {"fillColor": "transparent", "color": "#fc0303",
                              "weight": 1, "fillOpacity": 0},
).add_to(fm)

# AOI (thick red)
folium.GeoJson(
    fset_to_geojson(aoi), name="CUP AOI",
    style_function=lambda x: {"fillColor": "transparent", "color": "#fc0303",
                              "weight": 3, "fillOpacity": 0},
).add_to(fm)

# Controls
folium.LayerControl(collapsed=True).add_to(fm)

# Floating analysis panel (bottom-left)
panel_html = f"""
<div style="position: fixed; bottom: 24px; left: 12px; z-index: 1000;
            background: rgba(255,255,255,0.96); padding: 12px 16px;
            border-radius: 6px; box-shadow: 0 2px 8px rgba(0,0,0,0.15);
            font-family: -apple-system, BlinkMacSystemFont, sans-serif;
            font-size: 13px; min-width: 220px; color: #222;">
  <div style="font-weight: 600; border-bottom: 1px solid #ccc;
              padding-bottom: 4px; margin-bottom: 6px;">
    {CUP_name}
  </div>
  <table style="border-collapse: collapse; width: 100%;">
    <tr><td style="padding: 3px 0;">CUP area</td>
        <td style="text-align: right; font-variant-numeric: tabular-nums;">{results['aoi_area_acres']:,.2f} ac</td></tr>
    <tr><td style="padding: 3px 0;">Farmland</td>
        <td style="text-align: right; font-variant-numeric: tabular-nums;">{results['farmland_area_acres']:,.2f} ac</td></tr>
    <tr style="border-top: 1px solid #ccc;">
        <td style="padding: 3px 0;">Coverage</td>
        <td style="text-align: right; font-variant-numeric: tabular-nums; font-weight: 600;">{results['pct_coverage']:.2f}%</td></tr>
  </table>
</div>
"""
fm.get_root().html.add_child(folium.Element(panel_html))

# Title in top-left
title_html = f"""
<div style="position: fixed; top: 12px; left: 60px; z-index: 1000;
            background: rgba(255,255,255,0.96); padding: 8px 14px;
            border-radius: 6px; box-shadow: 0 2px 8px rgba(0,0,0,0.15);
            font-family: -apple-system, BlinkMacSystemFont, sans-serif;
            font-size: 14px; font-weight: 600; color: #222;">
  {CUP_number} Farmland Analysis
</div>
"""
fm.get_root().html.add_child(folium.Element(title_html))

fm

In [ ]:
output_path = f"{CUP_number}_map.html"
fm.save(output_path)

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print(f"Saved to {output_path}")